In [1]:
import numpy as np 
import pandas as pd 
import re 
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [4]:
twitter_data = pd.read_csv(
    "training.1600000.processed.noemoticon.csv",
    encoding="ISO-8859-1",
    skiprows=range(1, 750000),
    nrows=100000
)

In [5]:
twitter_data.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,2285370474,Mon Jun 22 15:02:48 PDT 2009,NO_QUERY,idmoore,"@Opotopo small slip on Tryfan few weeks back, ..."
1,0,2285370823,Mon Jun 22 15:02:49 PDT 2009,NO_QUERY,xbeautifulmessx,@Idristwilight You can post HAN when you want....
2,0,2285371185,Mon Jun 22 15:02:51 PDT 2009,NO_QUERY,thefirstsight,@rose_7 Ohh poor jan please tell her that if ...
3,0,2285371495,Mon Jun 22 15:02:52 PDT 2009,NO_QUERY,Sarah2713,Finally home from work...It was a looong day!!...
4,0,2285371762,Mon Jun 22 15:02:54 PDT 2009,NO_QUERY,dierockerfrau,im very sad 4 chantelle and tom


In [6]:
twitter_data.shape

(100000, 6)

In [7]:
#naming the columns in dataframe
column_name = ['target', 'id', 'data', 'flag', 'user', 'text']
twitter_data = pd.read_csv(
    "training.1600000.processed.noemoticon.csv",
    encoding="ISO-8859-1",
    names = column_name,
    skiprows=range(1, 750000),
    nrows=100000
)

In [8]:
twitter_data.head()

,target,id,data,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,2285370474,Mon Jun 22 15:02:48 PDT 2009,NO_QUERY,idmoore,"@Opotopo small slip on Tryfan few weeks back, ..."
2,0,2285370823,Mon Jun 22 15:02:49 PDT 2009,NO_QUERY,xbeautifulmessx,@Idristwilight You can post HAN when you want....
3,0,2285371185,Mon Jun 22 15:02:51 PDT 2009,NO_QUERY,thefirstsight,@rose_7 Ohh poor jan please tell her that if ...
4,0,2285371495,Mon Jun 22 15:02:52 PDT 2009,NO_QUERY,Sarah2713,Finally home from work...It was a looong day!!...


In [9]:
twitter_data.shape

(100000, 6)

In [ ]:
twitter_data['target'].value_counts()

In [10]:
twitter_data.replace({4:1},inplace=True)

In [11]:
neg = twitter_data[twitter_data['target'] == 0]
pos = twitter_data[twitter_data['target'] == 1]

In [12]:
neg_sample = neg.sample(5000, random_state=42)
pos_sample = pos.sample(5000, random_state=42)

In [13]:
twitter_data = pd.concat([neg_sample, pos_sample])

In [14]:
twitter_data = twitter_data.sample(frac=1, random_state=42).reset_index(drop=True)

In [15]:
twitter_data.isnull().sum()

target    0
id        0
data      0
flag      0
user      0
text      0
dtype: int64

In [17]:
twitter_data.replace({4:1},inplace=True)

In [18]:
twitter_data['target'].value_counts()

target
1    5000
0    5000
Name: count, dtype: int64

In [19]:
port_stem = PorterStemmer()
def stemming(content):
    stremmed_content = re.sub("[^a-zA-Z]", " ", content)
    stremmed_content = stremmed_content.lower()
    stremmed_content = stremmed_content.split()
    stremmed_content = [port_stem.stem(word) for word in stremmed_content if not word is stopwords.words('english')]
    stremmed_content = ' '.join(stremmed_content)

    return stremmed_content

In [21]:
twitter_data['stremmed_content'] = twitter_data['text'].apply(stemming) 

In [22]:
twitter_data.head()

,target,id,data,flag,user,text,stremmed_content
0,1,1563677738,Sun Apr 19 23:47:17 PDT 2009,NO_QUERY,angela_bee,"Rice Prayer Meeting with ness, jen, sam, olly....",rice prayer meet with ness jen sam olli one hu...
1,0,2323149358,Thu Jun 25 00:00:49 PDT 2009,NO_QUERY,aznJaime,@camerontdf @ericjtdf @davidptdf You guys have...,camerontdf ericjtdf davidptdf you guy have no ...
2,0,2300750659,Tue Jun 23 14:32:44 PDT 2009,NO_QUERY,bellekid,Made seven layer bars with walnuts for a party...,made seven layer bar with walnut for a parti t...
3,0,2325425727,Thu Jun 25 05:25:37 PDT 2009,NO_QUERY,Dutchrudder,@artyjill really train on a orange ?? Lol it's...,artyjil realli train on a orang lol it s somth...
4,0,2324078436,Thu Jun 25 02:13:47 PDT 2009,NO_QUERY,peeweetheo,cant find my eurotrip dvd,cant find my eurotrip dvd


In [29]:
X = twitter_data['stremmed_content']

In [30]:
Y= twitter_data['target']

In [36]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    stratify=y,
    random_state=2
)

In [37]:
print(X_train.shape)

(8000,)


In [38]:
print(X_test.shape)

(2000,)


In [39]:
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)

Y_train.to_csv("Y_train.csv", index=False)
Y_test.to_csv("Y_test.csv", index=False)